# 02 — Modello: baseline vs XGBoost

Domande: XGBoost batte davvero la logistica? Le probabilità sono calibrate? Quale soglia usare, sapendo che concedere credito a un cattivo pagatore costa 5 volte più che rifiutare un buon pagatore?

Prima di eseguire: `python -m credit_xai.pipeline` dalla cartella del progetto.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from credit_xai import config
from credit_xai.data import load_processed
from credit_xai.features import get_X_y
from credit_xai.model import CreditModel

sns.set_theme(style='whitegrid')
train, test = load_processed('train'), load_processed('test')
X_test, y_test = get_X_y(test)
model = CreditModel.load()

## 1. Metriche salvate dal training

In [ ]:
metrics = json.loads(config.METRICS_FILE.read_text())
pd.DataFrame({k: {m: v for m, v in d.items() if not isinstance(v, dict)}
              for k, d in metrics.items()})

## 2. Curva ROC: logistica vs XGBoost

In [ ]:
from sklearn.metrics import RocCurveDisplay
from credit_xai.train import build_logreg

X_train, y_train = get_X_y(train)
logreg = build_logreg().fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_predictions(y_test, logreg.predict_proba(X_test)[:, 1], name='Logistica', ax=ax)
RocCurveDisplay.from_predictions(y_test, model.prob_default(X_test), name='XGBoost', ax=ax)
ax.plot([0, 1], [0, 1], ls='--', c='grey')
ax.set_title('ROC sul test set')

## 3. Calibrazione

Se il modello dice 30%, circa il 30% di quei clienti va in default? Conta perché la probabilità viene mostrata al cliente e all'analista.

In [ ]:
from sklearn.calibration import CalibrationDisplay

fig, ax = plt.subplots(figsize=(6, 5))
CalibrationDisplay.from_predictions(y_test, logreg.predict_proba(X_test)[:, 1], n_bins=8, name='Logistica', ax=ax)
CalibrationDisplay.from_predictions(y_test, model.prob_default(X_test), n_bins=8, name='XGBoost', ax=ax)
ax.set_title('Curva di calibrazione')

## 4. Costo atteso al variare della soglia

In [ ]:
import numpy as np
from credit_xai.train import expected_cost

proba = model.prob_default(X_test)
grid = np.linspace(0.05, 0.95, 91)
costs = [expected_cost(y_test, (proba >= t).astype(int)) for t in grid]
plt.figure(figsize=(7, 4))
plt.plot(grid, costs)
plt.axvline(model.threshold, ls='--', c='red', label=f'soglia scelta su train ({model.threshold:.2f})')
plt.axvline(0.5, ls=':', c='grey', label='soglia standard 0.5')
plt.xlabel('soglia sulla probabilità di default'); plt.ylabel('costo medio per richiesta')
plt.legend(); plt.title('Perché non usare 0.5')

## 5. Matrice di confusione con la soglia scelta

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    y_test, model.predict(X_test),
    display_labels=['buon pagatore', 'cattivo pagatore'], cmap='Blues')
plt.title('Righe: realtà · colonne: decisione (1 = rifiuto)')

## Note

- Differenza di AUC tra i due modelli: …
- Calibrazione: …
- Effetto della soglia sul tasso di approvazione: …